# Entrega da Programação para a STS

Gera um arquivo `.txt` com as atividades solicitadas, sobe ao SharePoint e registra
metadados em `lake_relatorios_gerados.dbo.prog_enviada`.

**Fonte:** `lake_gold_fatos.dbo.base`  
**Entrada:** `atividade_ids_str` (CSV, injetado pelo Power Automate)  
**Saída:** arquivo `.txt` no SharePoint + linha na tabela `prog_enviada`

In [ ]:
# ── Parâmetros (Power Automate injeta valores em produção) ─────────────────
atividade_ids_str   = '63000016479883,63000018896693,63000018461299'

SHAREPOINT_SITE_URL = 'https://sescsp.sharepoint.com/sites/GTDadosSTS'
SHAREPOINT_FOLDER   = '/Shared Documents/General/UO_para_STS'
SHAREPOINT_LIBRARY  = 'Documents'
solicitante         = ''   # email de quem solicitou (injetado pelo PA futuramente)
tipo_relatorio      = 'Entrega da Programação para a STS'

# Credenciais do Service Principal para Graph API (mesmas do PA)
GRAPH_TENANT_ID     = ''   # ex: 'xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx'
GRAPH_CLIENT_ID     = ''   # client_id do app registration
GRAPH_CLIENT_SECRET = ''   # client_secret

In [ ]:
import json
import struct
import re
import uuid
import tempfile
import warnings
from datetime import datetime
from pathlib import Path

import pandas as pd
import requests

warnings.filterwarnings('ignore')

# ── Detecção de ambiente ───────────────────────────────────────────────────
try:
    spark
    FABRIC_ENV = True
    print('Ambiente: Microsoft Fabric')
except NameError:
    FABRIC_ENV = False
    print('Ambiente: local')

try:
    from notebookutils import mssparkutils as _ms
    mssparkutils = _ms
    HAS_MSSPARKUTILS = True
except ImportError:
    HAS_MSSPARKUTILS = False

if not FABRIC_ENV:
    try:
        import pyodbc
        from azure.identity import InteractiveBrowserCredential, DeviceCodeCredential
        _HAS_PYODBC = True
    except ImportError:
        _HAS_PYODBC = False
        print('[AVISO] pyodbc / azure-identity nao disponiveis.')
else:
    _HAS_PYODBC = False

SQL_ENDPOINT = (
    'beu5bmmdbuwedpv62ucm524jzi-dmrv7k3fbwbevh5d4sidg3urfq'
    '.datawarehouse.fabric.microsoft.com'
)

print('Imports OK -', datetime.now().strftime('%d/%m/%Y %H:%M'))

In [ ]:
# ── Helpers de autenticacao ───────────────────────────────────────────────

def _get_db_conn():
    try:
        cred = InteractiveBrowserCredential()
    except Exception:
        cred = DeviceCodeCredential()
    token = cred.get_token('https://database.windows.net/.default').token
    tb = token.encode('utf-16-le')
    ts = struct.pack(f'<I{len(tb)}s', len(tb), tb)
    return pyodbc.connect(
        f'DRIVER={{ODBC Driver 17 for SQL Server}};'
        f'SERVER={SQL_ENDPOINT};Encrypt=Yes;',
        attrs_before={1256: ts},
    )


def _get_graph_token() -> str:
    # Fabric: usa ClientSecretCredential com o SP do Power Automate
    # (mssparkutils.credentials.getToken falha para Graph API sem permissões explícitas)
    if HAS_MSSPARKUTILS and GRAPH_CLIENT_SECRET:
        from azure.identity import ClientSecretCredential
        cred = ClientSecretCredential(GRAPH_TENANT_ID, GRAPH_CLIENT_ID, GRAPH_CLIENT_SECRET)
        return cred.get_token('https://graph.microsoft.com/.default').token
    # Fallback: tenta via mssparkutils (requer workspace com permissão Graph configurada)
    if HAS_MSSPARKUTILS:
        return mssparkutils.credentials.getToken('https://graph.microsoft.com')
    try:
        cred = InteractiveBrowserCredential()
    except Exception:
        cred = DeviceCodeCredential()
    return cred.get_token('https://graph.microsoft.com/.default').token


print('Helpers de autenticacao definidos.')

In [ ]:
# ── Busca atividade_id, nome, custo_total e gerencia em lake_gold_fatos ────

def fetch_atividades(ids: list) -> pd.DataFrame:
    q = chr(39)  # aspas simples para montar IN clause no Spark
    ids_sql = ','.join(q + str(i) + q for i in ids)
    sql = (
        'SELECT atividade_id, nome, custo_total, gerencia '
        'FROM lake_gold_fatos.dbo.base '
        f'WHERE atividade_id IN ({ids_sql})'
    )
    if FABRIC_ENV:
        df = spark.sql(sql).toPandas()
    else:
        # Localmente: pyodbc com IDs numéricos (sem aspas) — SQL endpoint não suporta aspas aqui
        ids_sql_local = ', '.join(str(int(i)) for i in ids)
        sql_local = (
            'SELECT atividade_id, nome, custo_total, gerencia '
            'FROM lake_gold_fatos.dbo.base '
            f'WHERE atividade_id IN ({ids_sql_local})'
        )
        conn = _get_db_conn()
        df = pd.read_sql(sql_local, conn)
        conn.close()

    df['atividade_id'] = pd.to_numeric(df['atividade_id'], errors='coerce').astype('Int64')
    df['custo_total']  = pd.to_numeric(df['custo_total'],  errors='coerce').fillna(0.0)
    return df


print('fetch_atividades definida.')

In [ ]:
# ── Geracao do arquivo .txt ───────────────────────────────────────────────

def gerar_txt(df: pd.DataFrame, path: Path) -> None:
    sep = chr(9)   # tab
    nl  = chr(10)  # newline
    header = sep.join(['atividade_id', 'nome', 'custo_total'])
    rows = [
        sep.join([str(row.atividade_id), str(row.nome), f'{row.custo_total:.2f}'])
        for row in df.itertuples(index=False)
    ]
    path.write_text(nl.join([header] + rows), encoding='utf-8')


print('gerar_txt definida.')

In [ ]:
# ── Upload para SharePoint via Microsoft Graph API ────────────────────────

def upload_sharepoint(local_path: Path, token: str) -> str:
    auth_json = {'Authorization': 'Bearer ' + token, 'Content-Type': 'application/json'}

    # 1. Resolve site_id
    hostname  = re.sub(r'https?://', '', SHAREPOINT_SITE_URL).split('/')[0]
    site_path = '/'.join(SHAREPOINT_SITE_URL.split('/')[3:])
    r = requests.get(
        'https://graph.microsoft.com/v1.0/sites/' + hostname + ':/' + site_path,
        headers=auth_json,
    )
    r.raise_for_status()
    site_id = r.json()['id']

    # 2. Resolve drive_id pela biblioteca
    drives_resp = requests.get(
        'https://graph.microsoft.com/v1.0/sites/' + site_id + '/drives',
        headers=auth_json,
    )
    drives_resp.raise_for_status()
    drive = next(
        d for d in drives_resp.json()['value']
        if d['name'] == SHAREPOINT_LIBRARY
    )
    drive_id = drive['id']

    # 3. Monta caminho relativo à raiz da biblioteca (Graph API não inclui o nome da lib)
    # SHAREPOINT_FOLDER pode vir como '/Shared Documents/General/...' ou '/General/...'
    folder = SHAREPOINT_FOLDER.lstrip('/')
    for prefix in ('Shared Documents/', SHAREPOINT_LIBRARY + '/'):
        if folder.startswith(prefix):
            folder = folder[len(prefix):]
            break
    remote_path = folder + '/' + local_path.name

    # 4. Upload via PUT (arquivos < 4 MB)
    up_resp = requests.put(
        'https://graph.microsoft.com/v1.0/drives/' + drive_id + '/root:/' + remote_path + ':/content',
        headers={'Authorization': 'Bearer ' + token, 'Content-Type': 'application/octet-stream'},
        data=local_path.read_bytes(),
    )
    up_resp.raise_for_status()
    return up_resp.json()['webUrl']


print('upload_sharepoint definida.')

In [ ]:
# ── Registro em lake_relatorios_gerados.dbo.prog_enviada ─────────────────

def registrar_relatorio(relatorio_id: str, relatorio_nome: str, url: str,
                        ids: list, qt_total: int, gerencias: str,
                        solicitante: str) -> None:
    row = pd.DataFrame([{
        'relatorio_id':   relatorio_id,
        'relatorio_nome': relatorio_nome,
        'data_geracao':   datetime.now(),
        'url_arquivo':    url,
        'solicitante':    solicitante,
        'atividade_ids':  ','.join(str(i) for i in ids),
        'qt_total':       qt_total,
        'gerencias':      gerencias,
    }])

    if FABRIC_ENV:
        (spark.createDataFrame(row)
              .write.mode('append')
              .option('mergeSchema', 'true')
              .saveAsTable('lake_relatorios_gerados.dbo.prog_enviada'))
        print('Linha registrada em lake_relatorios_gerados.dbo.prog_enviada.')
    else:
        print('LOCAL - registro que seria gravado em lake_relatorios_gerados.dbo.prog_enviada:')
        print(row.to_string(index=False))


print('registrar_relatorio definida.')

In [ ]:
# ── Execucao principal ────────────────────────────────────────────────────

# Normaliza o input: adiciona colchetes se veio sem eles ({...},{...} → [{...},...])
raw = atividade_ids_str.strip()
if raw.startswith('{'):
    raw = '[' + raw + ']'

# Aceita JSON array ('[{"atividade_id":123},...]' ou '[123,456]') ou CSV ("123,456")
if raw.startswith('['):
    parsed = json.loads(raw)
    if parsed and isinstance(parsed[0], dict):
        ids = [int(x['atividade_id']) for x in parsed]
    else:
        ids = [int(x) for x in parsed]
else:
    ids = [int(x.strip()) for x in raw.split(',') if x.strip()]

print(f'{len(ids)} atividades: {ids}')

# 1. Consulta
df = fetch_atividades(ids)
print(f'Dados retornados: {len(df)} linha(s)')
print(df.to_string(index=False))

# 2. Gera .txt
ts_str   = datetime.now().strftime('%Y%m%d_%H%M%S')
filename = f'relatorio_sts_{ts_str}.txt'
tmp_path = Path(tempfile.gettempdir()) / filename
gerar_txt(df, tmp_path)
print(f'Arquivo gerado: {tmp_path}')

# 3. Upload para SharePoint
graph_token = _get_graph_token()
url = upload_sharepoint(tmp_path, graph_token)
print(f'Arquivo enviado ao SharePoint: {url}')

# 4. Registra metadados
rid       = str(uuid.uuid4())
gerencias = '|'.join(sorted(df['gerencia'].dropna().unique()))
registrar_relatorio(rid, tipo_relatorio, url, ids, len(ids), gerencias, solicitante)

# 5. Retorno para o Power Automate
resultado = json.dumps({
    'relatorio_id':   rid,
    'relatorio_nome': tipo_relatorio,
    'url':            url,
    'qt_total':       len(ids),
    'gerencias':      gerencias,
    'gerado_em':      datetime.now().isoformat(),
}, ensure_ascii=False)
print(resultado)

if HAS_MSSPARKUTILS:
    mssparkutils.notebook.exit(resultado)